# 05 — LLM Agent (Evidence-Traced Graph QA)

**Pipeline Step 5 of 5**

This notebook demonstrates the LLM-powered QA agent that queries the Micro-CKG with strict evidence traceability. Every answer cites exact `(Source)--[Edge_Type, Score=X.XX]-->(Target)` graph evidence.

### Hardened Guardrails
1. **No External Knowledge** — answers from graph context only
2. **Missing Data Fallback** — explicit "No evidence found" response
3. **Mandatory Citation** — `[Evidence: (Source) --(Edge)--> (Target)]`
4. **Objective Tone** — no speculation

### Prerequisites
- `GOOGLE_API_KEY` set in `.env` file (gitignored)

### Inputs
| File | Description |
|---|---|
| `cache/micro_ckg.graphml` | Serialized Micro-CKG from Step 04 |

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from src.biocypher_adapter import load_graph
from src.llm_agent import create_qa_agent, query_graph

CACHE_DIR = PROJECT_ROOT / "cache"

print("Imports ready.")

Imports ready.


## 5.1 Load Micro-CKG

We load the persisted GraphML file produced by notebook 04. The graph is deserialized back into a NetworkX DiGraph with all node and edge attributes intact. The summary below confirms the graph structure matches expectations (number of gene nodes, cell-type nodes, region nodes, and total edges).

In [3]:
graph = load_graph(CACHE_DIR / "micro_ckg.graphml")
print(f"\nMicro-CKG: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

  Graph loaded: 45 nodes, 471 edges

Micro-CKG: 45 nodes, 471 edges


## 5.2 Create QA Agent (Google Gemini 2.0 Flash)

The QA agent uses **Google Gemini 2.0 Flash** for fast, accurate inference with a 1M-token context window. The API key is loaded from `.env` (gitignored). The system prompt contains the serialized graph data and strict evidence-traceability rules.

To switch providers, change `provider` to `"openai"` or `"ollama"` and set the corresponding API key.

In [4]:
agent = create_qa_agent(graph, provider="google")
print("QA agent initialised (Google Gemini 2.0 Flash).")

QA agent initialised (Google Gemini 2.0 Flash).


## 5.3 Killer Demo Queries

Three questions designed to demonstrate the agent's evidence-traceability guardrails:

1. **Biomarker Discovery** — What statistically significant genes drive the AD condition? Tests the agent's ability to cite exact DE evidence from graph edges.
2. **Spatial Context** — Which anatomical regions harbor key neuroinflammatory markers? Tests the agent's ability to traverse gene → cell-type → region paths.
3. **Anti-Hallucination Trap** — Ask about Tau tangles and Lecanemab (neither exists in the graph). Tests the "No evidence found" guardrail.

In [5]:
# Query 1: Biomarker Discovery
question1 = "Based on the Micro-CKG, what are the top statistically significant genes driving the AD (Disease) condition?"
answer1 = query_graph(agent, question1)
print(answer1)

  Querying agent: Based on the Micro-CKG, what are the top statistically significant genes driving...
  Rate-limited (attempt 1/6). Retrying in 30s...
  Rate-limited (attempt 2/6). Retrying in 60s...
  Rate-limited (attempt 3/6). Retrying in 120s...


KeyboardInterrupt: 

In [ ]:
# Query 2: Spatial Context
question2 = "Which specific spatial clusters or anatomical regions are these key neuroinflammatory markers (e.g., Fth1, Prnp, Calb1) mapped to in the graph?"
answer2 = query_graph(agent, question2)
print(answer2)

In [ ]:
# Query 3: Anti-Hallucination Trap
question3 = "Does the graph indicate any relationship between Tau tangle formation and the FDA-approved drug Lecanemab?"
answer3 = query_graph(agent, question3)
print(answer3)